In [ ]:
from datasets import load_dataset, concatenate_datasets

dataset_easy = load_dataset("dnth/ssf-dataset-synthetic-v4", "easy_triplets")
dataset_easy_v2 = load_dataset("dnth/ssf-dataset-synthetic-v4", "easy_triplets_v2")

dataset_hard = load_dataset("dnth/ssf-dataset-synthetic-v4", "hard_triplets")


In [ ]:
dataset = concatenate_datasets([dataset_easy["train"], dataset_easy_v2["train"], dataset_hard["train"]])
dataset

In [ ]:
dataset = (
    dataset.select_columns(["anchor", "positive", "negative"])
    .add_column("id", [str(i) for i in range(len(dataset))])
    .train_test_split(test_size=0.1, seed=42)
)
dataset

In [ ]:
# save datasets to disk
dataset["train"].to_json("train_dataset.json", orient="records")
dataset["test"].to_json("test_dataset.json", orient="records")

In [ ]:
from datasets import load_dataset, concatenate_datasets

test_dataset = load_dataset("json", data_files="test_dataset.json", split="train")
train_dataset = load_dataset("json", data_files="train_dataset.json", split="train")
corpus_dataset = concatenate_datasets([train_dataset, test_dataset])

In [ ]:
# Convert the datasets to dictionaries
corpus = dict(
    zip(corpus_dataset["id"], corpus_dataset["positive"])
)  # Our corpus (cid => document)
queries = dict(
    zip(test_dataset["id"], test_dataset["anchor"])
)  # Our queries (qid => question)

# Create a mapping of relevant document (1 in our case) for each query
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for q_id in queries:
    relevant_docs[q_id] = [q_id]

In [ ]:
from sentence_transformers.util import cos_sim
from sentence_transformers.evaluation import InformationRetrievalEvaluator


evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="default",
    score_functions={"cosine": cos_sim},
)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('nomic-ai/modernbert-embed-base')

results = evaluator(model)

In [ ]:
results

In [ ]:
model = SentenceTransformer('dnth/ssf-retriever-modernbert-embed-base-v3.2')

results = evaluator(model)
results